# Groundedness feedback check
1. Create ground truth answers based on authoritative sources (keyword searching from Opensearch)
2. Run the agent based on the same question and compare its retrieved results

In [3]:
import sys
from pathlib import Path
import requests

print(f"Python Version: {sys.version_info.major}.{sys.version_info.minor}.{sys.version_info.micro}")
print(f"Environment: {sys.executable}")

current_dir = Path.cwd()
if current_dir.name == "notebooks":
    project_root = current_dir.parent
else:
    project_root = current_dir

print(f"Project Root: {project_root}")

if project_root and (project_root / "src").exists():
    sys.path.insert(0, str(project_root))
else:
    print("Project root not found or src directory missing")
    sys.exit(1)


Python Version: 3.12.11
Environment: /Users/xieqiqi/Learning/LLM/my_first_Rag_project/.venv/bin/python3
Project Root: /Users/xieqiqi/Learning/LLM/my_first_Rag_project


In [9]:
# Simple BM25 Search
from src.services.opensearch.factory import make_opensearch_client

# search_term = "What is the latest model for image classification"

search_term = "arXiv ID: 2603.22279v1"
print(f"Searching for {search_term}")

opensearch_client = make_opensearch_client()
results = opensearch_client.search_papers(
    query=search_term,
    size=5
)
if results.get('hits'):
    print(f"Found {results.get('total', 0)} total matches\n")
    
    for i, paper in enumerate(results['hits'], 1):
        print(f"{i}. {paper.get('title', 'Unknown')[:70]}...")
        print(f"   Score: {paper.get('score', 0):.2f}")
        print(f"   arXiv ID: {paper.get('arxiv_id', 'N/A')}\n")
else:
    print("No results found. Try searching for:")
    print("  • 'neural', 'model', 'algorithm'")
    print("  • Use '*' to see all papers")


Searching for arXiv ID: 2603.22279v1
Found 44 total matches

1. 3D-Layout-R1: Structured Reasoning for Language-Instructed Spatial Edi...
   Score: 13.25
   arXiv ID: 2603.22279v1

2. 3D-Layout-R1: Structured Reasoning for Language-Instructed Spatial Edi...
   Score: 6.08
   arXiv ID: 2603.22279v1

3. 3D-Layout-R1: Structured Reasoning for Language-Instructed Spatial Edi...
   Score: 6.02
   arXiv ID: 2603.22279v1

4. 3D-Layout-R1: Structured Reasoning for Language-Instructed Spatial Edi...
   Score: 5.75
   arXiv ID: 2603.22279v1

5. 3D-Layout-R1: Structured Reasoning for Language-Instructed Spatial Edi...
   Score: 5.07
   arXiv ID: 2603.22279v1



In [6]:
# Hybrid search

print("Test search functionality")
print("="*20)

search_query = "What is the latest model for image classification"
print(f"Searching for: {search_query}")

try:
    search_request = {
        "query": search_query,
        "use_hybrid": True,
        "size": 5
    }
    response = requests.post(
        "http://localhost:8000/api/v1/hybrid-search",
        json=search_request,
        timeout=30 
    )

    if response.status_code == 200:
        data = response.json()
        print(f"✓ Found {data['total']} results")
        print(f"✓ Search mode: {data['search_mode']}")
        
        if data['hits']:
            print("\nTop results:")
            for i, hit in enumerate(data['hits'][:5], 1):
                title = hit.get('title', 'Unknown')[:60]
                score = hit.get('score', 0)
                print(f"  {i}. {title}... (score: {score:.3f})")
        else:
            print("No results found")
    else:
        print(f"✗ Search failed: {response.status_code}")
        
except Exception as e:
    print(f"✗ Error: {e}")

Test search functionality
Searching for: What is the latest model for image classification
✓ Found 5 results
✓ Search mode: hybrid

Top results:
  1. SHAPE: Structure-aware Hierarchical Unsupervised Domain Adap... (score: 0.016)
  2. End-to-End Training for Unified Tokenization and Latent Deno... (score: 0.016)
  3. SHAPE: Structure-aware Hierarchical Unsupervised Domain Adap... (score: 0.016)
  4. UniMotion: A Unified Framework for Motion-Text-Vision Unders... (score: 0.016)
  5. SHAPE: Structure-aware Hierarchical Unsupervised Domain Adap... (score: 0.016)


In [7]:
import time
import requests


question = "What is the latest model for image classification?"
print(f"Question: {question}")
print(f"Excepted: Guardrail should reject (score<60) and explain scope \n")

REQUEST_TIMEOUT = 300
TRUNCATE_ANSWERS = True
TRUNCATE_LENGTH = 200

start_time = time.time()
try:
    response = requests.post(
        "http://localhost:8000/api/v1/ask-agentic",
        json={
            "question": question,
            "top_k": 5,
            "use_hybrid": True,
        },
        timeout=REQUEST_TIMEOUT,
    )
    elapsed = time.time() - start_time
    if response.status_code == 200:
        data = response.json()
        print(f"✓ Agentic RAG ({elapsed:.1f}s)")
        print(f"\nAnswer: {data['answer']}")
        print(f"\nRetrieval attempts: {data.get('retrieval_attempts', 0)}")
        print(f"\nReasoning steps:")
        for i, step in enumerate(data.get('reasoning_steps', []), 1):
            print(f"  {i}. {step}")

        guardrail_step = next(
            (s for s in data.get('reasoning_steps', []) if 'validated' in s.lower() and 'score' in s.lower()),
            None
        )
        if guardrail_step:
            print(f"\nGuardrail validation: {guardrail_step}")
        if data.get("retrieval_attempts", 0) == 0:
            print("\n✓ Guardrail rejected out of scope question")
        else:
            print("\n✗ Guardrail did not reject out of scope question")
    else:
        print(f"✗ Agentic RAG failed (status code: {response.status_code})")
        print(f"Response:{response.text}")

except Exception as e:
    print(f"✗ Error: {e}")

Question: What is the latest model for image classification?
Excepted: Guardrail should reject (score<60) and explain scope 

✓ Agentic RAG (0.6s)

Answer: content=["I apologize, but I can only help with questions about academic research papers in Computer Science, Artificial Intelligence, and Machine Learning from arXiv.\n\nYour question: 'What is the latest model for image classification?'\n\nThis appears to be outside my domain of expertise. For questions like this, you might want to try:\n- General-purpose AI assistants for broad knowledge questions\n- Domain-specific resources for topics outside CS/AI/ML\n- Technical documentation if asking about specific software/tools\n\nIf you have a question about AI/ML research papers, I'd be happy to help!"] additional_kwargs={} response_metadata={} id='bd319334-94a6-4345-9f4a-134ff09a7f08' tool_calls=[] invalid_tool_calls=[]

Retrieval attempts: 0

Reasoning steps:
  1. Validated query scope (score: 50/100)
  2. Generated answer from contex